# Per-strain activity models — classifier comparison

For each of the five bacterial strains, four classifiers (MLP, SVM, random forest, XGBoost) are compared by 10-fold cross-validation at predicting activity against that strain. All models use the same preprocessing: **generations 0–4 only, duplicate sequences removed (265 sequences per strain)**, and the 11-bit L/D stereochemistry fingerprint. Hyperparameters are held fixed across strains so the comparison is fair; scrambled-label AUC is reported as a null control.

#### Imports

In [1]:
import numpy as np
import pandas as pd
from random import seed, sample

from sklearn.model_selection import StratifiedKFold
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score, accuracy_score, precision_score, recall_score, f1_score

#### Configuration

In [2]:
def seq_fingerprint(sequence):
    fp = np.zeros(len(sequence))
    for i, char in enumerate(sequence):
        if char.isupper():
            fp[i] = 1
    return fp

# strain display name -> data file (activity label column is auto-detected)
strains = {
    'E. coli W3110':          'data/activity-ecoli.xlsx',
    'P. aeruginosa PAO1':     'data/activity-paerug.xlsx',
    'A. baumannii ATCC19606': 'data/activity-abaum.xlsx',
    'K. pneumoniae NCTC418':  'data/activity-kpneu.xlsx',
    'MRSA (S. aureus COL)':   'data/activity-mrsa.xlsx',
}

# fresh estimator per fold; same hyperparameters across all strains for a fair comparison
models = {
    'MLP':     lambda: MLPClassifier(alpha=1, max_iter=1000, hidden_layer_sizes=(32, 32, 32), random_state=42),
    'SVM':     lambda: SVC(kernel='rbf', probability=True, random_state=42),
    'RF':      lambda: RandomForestClassifier(n_estimators=100, random_state=42),
    'XGBoost': lambda: XGBClassifier(n_estimators=100, eval_metric='logloss', random_state=42),
}

#### Load and preprocess each strain (rounds 0–4, deduplicated → 265 sequences)

In [3]:
def load_strain(path):
    df = pd.read_excel(path)
    # the activity label is the only column that is not Name / Sequence / round
    label_col = [c for c in df.columns if c not in ('Name', 'Sequence', 'round')][0]
    # rounds 0-4 only, drop duplicate sequences
    df = df[df['round'] <= 4].drop_duplicates(subset='Sequence').reset_index(drop=True)
    X = np.array([seq_fingerprint(s) for s in df['Sequence']])
    y = np.array(df[label_col].astype(int).tolist())
    return X, y

data = {}
for sname, path in strains.items():
    X, y = load_strain(path)
    data[sname] = (X, y)
    print(f'{sname:24s} n={len(y)}  actives={int(y.sum())}')

E. coli W3110            n=265  actives=209
P. aeruginosa PAO1       n=265  actives=124
A. baumannii ATCC19606   n=265  actives=144
K. pneumoniae NCTC418    n=265  actives=76
MRSA (S. aureus COL)     n=265  actives=114


#### Cross-validation helper

In [4]:
def cv_metrics(factory, X, y, cv):
    aucs, accs, precs, recs, f1s = [], [], [], [], []
    for train, test in cv.split(X, y):
        model = factory()
        model.fit(X[train], y[train])
        score = model.predict_proba(X[test])[:, 1]
        pred = model.predict(X[test])
        aucs.append(roc_auc_score(y[test], score))
        accs.append(accuracy_score(y[test], pred))
        precs.append(precision_score(y[test], pred, zero_division=0))
        recs.append(recall_score(y[test], pred, zero_division=0))
        f1s.append(f1_score(y[test], pred, zero_division=0))
    return {'AUC': np.mean(aucs), 'AUC_std': np.std(aucs),
            'accuracy': np.mean(accs), 'precision': np.mean(precs),
            'recall': np.mean(recs), 'F1': np.mean(f1s)}

#### Run the strain × model comparison (true and scrambled labels)

In [5]:
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

rows = []
for sname, (X, y) in data.items():
    seed(42)
    y_scrambled = np.array(sample(list(y), len(y)))
    for mname, factory in models.items():
        m = cv_metrics(factory, X, y, cv)
        auc_scr = cv_metrics(factory, X, y_scrambled, cv)['AUC']
        rows.append({'strain': sname, 'model': mname, 'n': len(y),
                     'AUC': round(m['AUC'], 2), 'AUC_std': round(m['AUC_std'], 2),
                     'AUC_scrambled': round(auc_scr, 2),
                     'accuracy': round(m['accuracy'], 2), 'precision': round(m['precision'], 2),
                     'recall': round(m['recall'], 2), 'F1': round(m['F1'], 2)})

results = pd.DataFrame(rows)
results

,strain,model,n,AUC,AUC_std,AUC_scrambled,accuracy,precision,recall,F1
0,E. coli W3110,MLP,265,0.84,0.06,0.55,0.81,0.85,0.93,0.89
1,E. coli W3110,SVM,265,0.78,0.10,0.52,0.79,0.80,0.99,0.88
2,E. coli W3110,RF,265,0.69,0.10,0.53,0.79,0.81,0.96,0.88
3,E. coli W3110,XGBoost,265,0.79,0.10,0.53,0.79,0.83,0.92,0.87
4,P. aeruginosa PAO1,MLP,265,0.81,0.10,0.50,0.77,0.79,0.69,0.73
5,P. aeruginosa PAO1,SVM,265,0.79,0.11,0.44,0.74,0.77,0.65,0.70
6,P. aeruginosa PAO1,RF,265,0.77,0.12,0.46,0.69,0.68,0.70,0.68
7,P. aeruginosa PAO1,XGBoost,265,0.77,0.11,0.52,0.71,0.70,0.69,0.69
8,A. baumannii ATCC19606,MLP,265,0.91,0.04,0.59,0.82,0.84,0.82,0.83
9,A. baumannii ATCC19606,SVM,265,0.88,0.06,0.61,0.80,0.82,0.80,0.81


#### AUC summary (strain × model)

Saved to `output/bacteria_model_comparison.csv`.

In [6]:
auc_table = results.pivot(index='strain', columns='model', values='AUC')[['MLP', 'SVM', 'RF', 'XGBoost']]
auc_table = auc_table.reindex(list(strains.keys()))
results.to_csv('output/bacteria_model_comparison.csv', index=False)
auc_table

model,MLP,SVM,RF,XGBoost
strain,,,,
E. coli W3110,0.84,0.78,0.69,0.79
P. aeruginosa PAO1,0.81,0.79,0.77,0.77
A. baumannii ATCC19606,0.91,0.88,0.90,0.90
K. pneumoniae NCTC418,0.90,0.87,0.86,0.85
MRSA (S. aureus COL),0.89,0.84,0.85,0.86
